# Lab 4_5: Pedestrian Detection Scenarios Using SIFT and HOG

**Course:** Computer Vision Lab (AI441L)  
**Task:** Comparison analysis of two pedestrian-detection scenarios on static images.

| Scenario | Technique | What we do |
|---|---|---|
| 1 | **SIFT** | Detect keypoints in two images of a pedestrian area, match them, verify the matches geometrically (RANSAC) |
| 2 | **HOG + linear SVM** | Detect pedestrians in both images with OpenCV's pre-trained people detector |

After the two scenarios, the notebook runs **robustness experiments** (brightness, blur, noise, scale, rotation, viewpoint change) and builds a **comparison table**.
Every figure is saved to the `figures/` folder, and the last cell zips them so you can paste them into the Word report.

**How to use:** Runtime → Run all. In step 2 you will be asked to upload **two images** of the same pedestrian area (ideally taken from different viewpoints, since SIFT matching needs overlapping content). If you skip the upload, the notebook downloads a demo image and creates a second viewpoint synthetically, so the code can still be run end to end.

In [ ]:
pip install 'opencv-python-headless<5'

## 1. Setup

In [ ]:
import os, time, urllib.request
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog as skimage_hog
from skimage import exposure

print("OpenCV version:", cv2.__version__)
assert hasattr(cv2, "SIFT_create"), "SIFT not available - use OpenCV >= 4.4"
assert hasattr(cv2, "HOGDescriptor"), "HOGDescriptor not available - use OpenCV 4.x (pip install 'opencv-python-headless<5')"

os.makedirs("figures", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11})
np.random.seed(0)

def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def save_fig(name):
    plt.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight")
    print(f"saved figures/{name}.png")

RESULTS = {}   # collects numbers for the report

OpenCV version: 5.0.0


AssertionError: HOGDescriptor not available - use OpenCV 4.x (pip install 'opencv-python-headless<5')

## 2. Load the two images

Upload **two** pedestrian-area images when prompted (JPG/PNG). Images wider than `MAX_WIDTH` are resized so that SIFT and HOG run quickly.  
Image 1 and Image 2 should show the **same scene from different viewpoints / conditions** (different angle, distance or lighting).

In [ ]:
MAX_WIDTH = 800          # resize wide images for speed
DEMO_URL = "https://raw.githubusercontent.com/opencv/opencv_extra/4.x/testdata/gpu/hog/road.png"

def load_and_resize(path):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Could not read {path}")
    h, w = img.shape[:2]
    if w > MAX_WIDTH:
        s = MAX_WIDTH / w
        img = cv2.resize(img, (MAX_WIDTH, int(h * s)), interpolation=cv2.INTER_AREA)
    return img

uploaded_paths = []
try:
    from google.colab import files
    print("Select TWO pedestrian images (hold Ctrl/Cmd to select both). Cancel to use the demo images.")
    up = files.upload()
    uploaded_paths = list(up.keys())
except Exception as e:
    print("Upload skipped:", e)

USING_DEMO = len(uploaded_paths) < 2
if not USING_DEMO:
    img1 = load_and_resize(uploaded_paths[0])
    img2 = load_and_resize(uploaded_paths[1])
    print("Using uploaded images:", uploaded_paths[:2])
else:
    print("Fewer than 2 images uploaded -> using DEMO images (image 2 is a synthetic new viewpoint).")
    urllib.request.urlretrieve(DEMO_URL, "demo_road.png")
    img1 = load_and_resize("demo_road.png")
    h, w = img1.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([[40, 30], [w - 20, 0], [w - 60, h - 20], [10, h - 50]])
    P = cv2.getPerspectiveTransform(src, dst)
    img2 = cv2.warpPerspective(img1, P, (w, h), borderMode=cv2.BORDER_REPLICATE)
    img2 = cv2.convertScaleAbs(img2, alpha=0.8, beta=-10)      # slightly darker

print("Image 1:", img1.shape, "| Image 2:", img2.shape)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(to_rgb(img1)); ax[0].set_title("Image 1"); ax[0].axis("off")
ax[1].imshow(to_rgb(img2)); ax[1].set_title("Image 2"); ax[1].axis("off")
plt.tight_layout(); save_fig("fig01_input_images"); plt.show()

## 3. Scenario 1 - SIFT keypoint detection and matching

**Pipeline**
1. Detect SIFT keypoints + 128-D descriptors in each image.
2. Match descriptors with a brute-force matcher (L2 distance), `k=2` nearest neighbours.
3. Keep a match only if it passes **Lowe's ratio test** (best distance < 0.75 x second-best distance).
4. Run **RANSAC** to fit a homography; matches consistent with it are *inliers* (geometrically verified correspondences).

In [ ]:
RATIO = 0.75          # Lowe ratio threshold
RANSAC_THR = 5.0      # pixels

def sift_match(a, b, ratio=RATIO, ransac_thr=RANSAC_THR):
    """Detect SIFT keypoints in a and b, match them and verify with RANSAC."""
    sift = cv2.SIFT_create()
    ga = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY)
    gb = cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)

    t0 = time.perf_counter()
    kpa, da = sift.detectAndCompute(ga, None)
    kpb, db = sift.detectAndCompute(gb, None)
    t_detect = time.perf_counter() - t0

    out = dict(kpa=kpa, kpb=kpb, good=[], mask=None, H=None, inliers=0,
               t_detect=t_detect, t_match=0.0, raw_matches=0)
    if da is None or db is None or len(kpa) < 2 or len(kpb) < 2:
        return out

    t0 = time.perf_counter()
    knn = cv2.BFMatcher(cv2.NORM_L2).knnMatch(da, db, k=2)
    good = []
    for pair in knn:
        if len(pair) == 2 and pair[0].distance < ratio * pair[1].distance:
            good.append(pair[0])
    out["t_match"] = time.perf_counter() - t0
    out["raw_matches"] = len(knn)
    out["good"] = good

    if len(good) >= 4:
        src = np.float32([kpa[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst = np.float32([kpb[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(src, dst, cv2.RANSAC, ransac_thr)
        if mask is not None:
            out["H"], out["mask"], out["inliers"] = H, mask.ravel().tolist(), int(mask.sum())
    return out

res = sift_match(img1, img2)
kp1, kp2, good = res["kpa"], res["kpb"], res["good"]

print(f"Keypoints   - Image 1: {len(kp1)} | Image 2: {len(kp2)}")
print(f"Ratio-test matches   : {len(good)}")
print(f"RANSAC inliers       : {res['inliers']}")
inlier_ratio = res["inliers"] / len(good) if good else 0
print(f"Inlier ratio         : {inlier_ratio:.2%}")
print(f"Detection time       : {res['t_detect']*1000:.0f} ms | Matching time: {res['t_match']*1000:.0f} ms")

RESULTS["sift"] = dict(kp1=len(kp1), kp2=len(kp2), good=len(good), inliers=res["inliers"],
                       inlier_ratio=inlier_ratio, t_detect_ms=res["t_detect"]*1000,
                       t_match_ms=res["t_match"]*1000)

### 3.1 Detected SIFT keypoints (circle size = scale, line = orientation)

In [ ]:
vis1 = cv2.drawKeypoints(img1, kp1, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
vis2 = cv2.drawKeypoints(img2, kp2, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].imshow(to_rgb(vis1)); ax[0].set_title(f"Image 1 - {len(kp1)} SIFT keypoints"); ax[0].axis("off")
ax[1].imshow(to_rgb(vis2)); ax[1].set_title(f"Image 2 - {len(kp2)} SIFT keypoints"); ax[1].axis("off")
plt.tight_layout(); save_fig("fig02_sift_keypoints"); plt.show()

### 3.2 Matches after Lowe's ratio test, and RANSAC-verified inliers

In [ ]:
# (a) all ratio-test matches (draw at most 150 for readability)
show = sorted(good, key=lambda m: m.distance)[:150]
vis_all = cv2.drawMatches(img1, kp1, img2, kp2, show, None,
                          flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
plt.figure(figsize=(14, 6))
plt.imshow(to_rgb(vis_all)); plt.axis("off")
plt.title(f"SIFT matches after ratio test ({len(good)} total, best {len(show)} drawn)")
plt.tight_layout(); save_fig("fig03_sift_matches_ratio_test"); plt.show()

# (b) RANSAC inliers only, plus the projected outline of image 1 in image 2
if res["mask"] is not None:
    inl = [m for m, k in zip(good, res["mask"]) if k]
    img2_outline = img2.copy()
    h1, w1 = img1.shape[:2]
    box = np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2)
    proj = cv2.perspectiveTransform(box, res["H"])
    cv2.polylines(img2_outline, [np.int32(proj)], True, (0, 255, 255), 3)
    vis_in = cv2.drawMatches(img1, kp1, img2_outline, kp2, sorted(inl, key=lambda m: m.distance)[:150], None,
                             matchColor=(0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(14, 6))
    plt.imshow(to_rgb(vis_in)); plt.axis("off")
    plt.title(f"RANSAC inliers: {len(inl)} / {len(good)} (yellow = Image 1 boundary projected into Image 2)")
    plt.tight_layout(); save_fig("fig04_sift_ransac_inliers"); plt.show()
else:
    print("Not enough matches to estimate a homography - try images with more overlap.")

## 4. Scenario 2 - HOG pedestrian detection

OpenCV's `HOGDescriptor` uses a **64 x 128 detection window**, 8 x 8 pixel cells, 16 x 16 blocks (stride 8) and 9 orientation bins, giving a 3780-D feature vector per window. A **pre-trained linear SVM** (Dalal-Triggs people detector) classifies each window while a sliding window runs over an image pyramid. Overlapping detections are then merged with non-maximum suppression (NMS).

**Why an `UPSCALE` parameter?** The detector only looks for people at least 64 x 128 px. In wide-angle scenes (typical for pedestrian areas) people are smaller, so we enlarge the image before detection and divide the box coordinates by the same factor afterwards.

In [ ]:
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

def iou(a, b):
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0

def overlap_min(a, b):
    """Intersection area divided by the smaller box area (catches boxes nested inside others)."""
    ax, ay, aw, ah = a; bx, by, bw, bh = b
    inter = max(0, min(ax + aw, bx + bw) - max(ax, bx)) * max(0, min(ay + ah, by + bh) - max(ay, by))
    return inter / max(1, min(aw * ah, bw * bh))

def nms(boxes, scores, thr=0.4, nest_thr=0.7):
    order = np.argsort(scores)[::-1]
    keep = []
    for i in order:
        if all(iou(boxes[i], boxes[j]) < thr and overlap_min(boxes[i], boxes[j]) < nest_thr for j in keep):
            keep.append(i)
    return [boxes[i] for i in keep], [scores[i] for i in keep]

def hog_detect(img, upscale=2.0, win_stride=(8, 8), padding=(8, 8), scale=1.05,
               hit_thr=0.0, score_thr=1.0, nms_thr=0.4):
    """Return (boxes in original image coordinates, scores, time in seconds)."""
    im = cv2.resize(img, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC) if upscale != 1 else img
    t0 = time.perf_counter()
    rects, weights = hog.detectMultiScale(im, hit_thr, win_stride, padding, scale, 2.0)
    dt = time.perf_counter() - t0
    if len(rects) == 0:
        return [], [], dt
    weights = np.array(weights).ravel()
    boxes = [tuple(int(round(v / upscale)) for v in r) for r in rects]
    keep = [i for i, s in enumerate(weights) if s >= score_thr]
    boxes = [boxes[i] for i in keep]; scores = [float(weights[i]) for i in keep]
    if boxes:
        boxes, scores = nms(boxes, scores, nms_thr)
    return boxes, scores, dt

def draw_boxes(img, boxes, scores, color=(0, 255, 0)):
    out = img.copy()
    for (x, y, w, h), s in zip(boxes, scores):
        cv2.rectangle(out, (x, y), (x + w, y + h), color, 2)
        cv2.putText(out, f"{s:.2f}", (x, max(12, y - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    return out

UPSCALE = 2.0        # enlarge image so small pedestrians reach the 64x128 window size
SCORE_THR = 1.0      # SVM score threshold (higher = fewer false positives, may miss people)

b1, s1, t1 = hog_detect(img1, upscale=UPSCALE, score_thr=SCORE_THR)
b2, s2, t2 = hog_detect(img2, upscale=UPSCALE, score_thr=SCORE_THR)
print(f"Image 1: {len(b1)} pedestrian(s) detected, scores = {np.round(s1, 2).tolist()}, time = {t1*1000:.0f} ms")
print(f"Image 2: {len(b2)} pedestrian(s) detected, scores = {np.round(s2, 2).tolist()}, time = {t2*1000:.0f} ms")

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
ax[0].imshow(to_rgb(draw_boxes(img1, b1, s1))); ax[0].set_title(f"Image 1 - HOG detections: {len(b1)}"); ax[0].axis("off")
ax[1].imshow(to_rgb(draw_boxes(img2, b2, s2))); ax[1].set_title(f"Image 2 - HOG detections: {len(b2)}"); ax[1].axis("off")
plt.tight_layout(); save_fig("fig06_hog_detections"); plt.show()

RESULTS["hog"] = dict(n1=len(b1), n2=len(b2), mean_score1=float(np.mean(s1)) if s1 else 0.0,
                      mean_score2=float(np.mean(s2)) if s2 else 0.0, t1_ms=t1*1000, t2_ms=t2*1000)

### 4.1 What does HOG actually see?
HOG summarises local gradient orientation. Below, the first detected pedestrian (or the image centre if nothing was detected) is resized to the 64 x 128 window and its HOG features are visualised: bright strokes show the dominant edge orientation in each cell.

In [ ]:
if b1:
    x, y, w, h = max(b1, key=lambda b: b[2] * b[3]); crop = img1[max(0, y):y + h, max(0, x):x + w]
else:
    H0, W0 = img1.shape[:2]; crop = img1[H0 // 4: H0 // 4 + 200, W0 // 2 - 50: W0 // 2 + 50]
crop64 = cv2.resize(crop, (64, 128))
gray64 = cv2.cvtColor(crop64, cv2.COLOR_BGR2GRAY)
feat, hog_img = skimage_hog(gray64, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
                            visualize=True, block_norm="L2-Hys")
hog_img = exposure.rescale_intensity(hog_img, in_range=(0, hog_img.max() + 1e-9))
print("HOG feature vector length for one 64x128 window:", feat.shape[0])

fig, ax = plt.subplots(1, 2, figsize=(6, 5))
ax[0].imshow(to_rgb(crop64)); ax[0].set_title("64x128 crop"); ax[0].axis("off")
ax[1].imshow(hog_img, cmap="gray"); ax[1].set_title("HOG visualisation"); ax[1].axis("off")
plt.tight_layout(); save_fig("fig05_hog_visualisation"); plt.show()

### 4.2 Optional - precision / recall against your own ground truth
If you want quantitative detection accuracy, annotate the pedestrians yourself as `(x, y, w, h)` boxes in the **resized** image coordinates (use the plots above to read approximate pixel positions) and paste them below. Leave the lists empty to skip.

In [ ]:
GT_IMG1 = []   # e.g. [(232, 190, 73, 145), (622, 157, 97, 194)]
GT_IMG2 = []

def prf(dets, gts, thr=0.3):
    matched, tp = set(), 0
    for d in dets:
        best, bj = 0, None
        for j, g in enumerate(gts):
            v = iou(d, g)
            if j not in matched and v > best:
                best, bj = v, j
        if best >= thr:
            tp += 1; matched.add(bj)
    fp, fn = len(dets) - tp, len(gts) - tp
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return tp, fp, fn, p, r, f

if GT_IMG1 or GT_IMG2:
    rows = []
    for name, dets, gts in (("Image 1", b1, GT_IMG1), ("Image 2", b2, GT_IMG2)):
        if gts:
            tp, fp, fn, p, r, f = prf(dets, gts)
            rows.append(dict(Image=name, TP=tp, FP=fp, FN=fn, Precision=round(p, 2), Recall=round(r, 2), F1=round(f, 2)))
    gt_table = pd.DataFrame(rows); display(gt_table)
    RESULTS["hog_gt"] = rows
else:
    print("No ground truth supplied - skipping precision/recall.")

### 4.3 HOG parameter study
How do the image up-scale factor, window stride, pyramid scale step and SVM score threshold change the number of detections and the run time? (Computed on Image 1.)

In [ ]:
rows = []
for up in (1.0, 1.5, 2.0, 2.5, 3.0):
    b, s, t = hog_detect(img1, upscale=up, score_thr=SCORE_THR)
    rows.append(dict(param="upscale", value=up, detections=len(b), time_ms=t * 1000))
for ws in ((4, 4), (8, 8), (16, 16)):
    b, s, t = hog_detect(img1, upscale=UPSCALE, win_stride=ws, score_thr=SCORE_THR)
    rows.append(dict(param="win_stride", value=f"{ws[0]}x{ws[1]}", detections=len(b), time_ms=t * 1000))
for sc in (1.01, 1.05, 1.1, 1.2):
    b, s, t = hog_detect(img1, upscale=UPSCALE, scale=sc, score_thr=SCORE_THR)
    rows.append(dict(param="pyramid_scale", value=sc, detections=len(b), time_ms=t * 1000))
for th in (0.0, 0.5, 1.0, 1.5, 2.0, 3.0):
    b, s, t = hog_detect(img1, upscale=UPSCALE, score_thr=th)
    rows.append(dict(param="score_thr", value=th, detections=len(b), time_ms=t * 1000))
sweep = pd.DataFrame(rows); sweep["time_ms"] = sweep["time_ms"].round(0)
display(sweep)

fig, axes = plt.subplots(2, 4, figsize=(15, 6))
for j, p in enumerate(["upscale", "win_stride", "pyramid_scale", "score_thr"]):
    d = sweep[sweep.param == p]
    axes[0, j].bar(d.value.astype(str), d.detections, color="#4C78A8"); axes[0, j].set_title(f"Detections vs {p}")
    axes[1, j].bar(d.value.astype(str), d.time_ms, color="#F58518"); axes[1, j].set_title(f"Time (ms) vs {p}")
    axes[1, j].set_xlabel(p)
plt.tight_layout(); save_fig("fig07_hog_parameter_study"); plt.show()
RESULTS["hog_sweep"] = sweep.to_dict("records")

## 5. Linking the two scenarios: SIFT keypoints on pedestrians
Scenario 1 states that SIFT keypoints can help establish the presence of pedestrians. Here we count how many SIFT keypoints fall **inside the HOG pedestrian boxes** versus the rest of the scene, and how many *matched/inlier* keypoints lie on pedestrians. Pedestrians move between viewpoints, so they are expected to be poorly matched, whereas static background structures (buildings, signs, kerbs) dominate the verified matches.

In [ ]:
def inside(pt, box):
    x, y, w, h = box
    return x <= pt[0] <= x + w and y <= pt[1] <= y + h

def count_in_boxes(kps, boxes):
    return sum(any(inside(k.pt, b) for b in boxes) for k in kps)

n_in1 = count_in_boxes(kp1, b1); n_in2 = count_in_boxes(kp2, b2)
area1 = sum(w * h for _, _, w, h in b1) / (img1.shape[0] * img1.shape[1])
area2 = sum(w * h for _, _, w, h in b2) / (img2.shape[0] * img2.shape[1])

inl_matches = [m for m, k in zip(good, res["mask"]) if k] if res["mask"] is not None else []
inl_on_ped = sum(1 for m in inl_matches if any(inside(kp1[m.queryIdx].pt, b) for b in b1))

link = pd.DataFrame([
    dict(Image="Image 1", keypoints=len(kp1), keypoints_in_pedestrian_boxes=n_in1,
         pct_keypoints_in_boxes=round(100 * n_in1 / max(1, len(kp1)), 1), pct_image_area_in_boxes=round(100 * area1, 1)),
    dict(Image="Image 2", keypoints=len(kp2), keypoints_in_pedestrian_boxes=n_in2,
         pct_keypoints_in_boxes=round(100 * n_in2 / max(1, len(kp2)), 1), pct_image_area_in_boxes=round(100 * area2, 1)),
])
display(link)
print(f"RANSAC inliers whose Image-1 keypoint lies on a detected pedestrian: {inl_on_ped} of {len(inl_matches)}")
RESULTS["link"] = dict(rows=link.to_dict("records"), inliers_on_ped=inl_on_ped, inliers_total=len(inl_matches))

vis = img1.copy()
for k in kp1:
    on_ped = any(inside(k.pt, b) for b in b1)
    cv2.circle(vis, (int(k.pt[0]), int(k.pt[1])), 2, (0, 0, 255) if on_ped else (255, 200, 0), -1)
vis = draw_boxes(vis, b1, s1)
plt.figure(figsize=(8, 6)); plt.imshow(to_rgb(vis)); plt.axis("off")
plt.title("SIFT keypoints: red = inside HOG pedestrian box, blue = elsewhere")
plt.tight_layout(); save_fig("fig08_sift_keypoints_on_pedestrians"); plt.show()

## 6. Robustness experiments (varying conditions)
We apply controlled changes to **Image 1** and measure how each method copes. Because we know the exact transformation `M` (a 3 x 3 matrix) we can evaluate both methods against ground truth:

* **SIFT** - match original vs. transformed image. *Correct-match precision* = fraction of ratio-test matches whose position agrees with `M` (error < 5 px).
* **HOG** - detect pedestrians in the transformed image. *Repeatability* = fraction of the original detections whose centre (mapped through `M`) lands inside a detection in the transformed image.

In [ ]:
def make_variants(img):
    h, w = img.shape[:2]
    I3 = np.eye(3)
    V = {}
    V["Original"] = (img.copy(), I3)
    V["Darker (x0.5)"] = (cv2.convertScaleAbs(img, alpha=0.5, beta=0), I3)
    V["Brighter (+60)"] = (cv2.convertScaleAbs(img, alpha=1.0, beta=60), I3)
    V["Blur (9x9)"] = (cv2.GaussianBlur(img, (9, 9), 0), I3)
    noisy = np.clip(img.astype(np.float32) + np.random.normal(0, 15, img.shape), 0, 255).astype(np.uint8)
    V["Noise (sigma=15)"] = (noisy, I3)
    for s in (0.5, 0.75, 1.5):
        M = np.diag([s, s, 1.0])
        V[f"Scale x{s}"] = (cv2.warpPerspective(img, M, (int(w * s), int(h * s))), M)
    R = cv2.getRotationMatrix2D((w / 2, h / 2), 20, 1.0)
    M = np.vstack([R, [0, 0, 1]])
    V["Rotate 20 deg"] = (cv2.warpPerspective(img, M, (w, h)), M)
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = np.float32([[60, 40], [w - 30, 0], [w - 90, h - 30], [20, h - 70]])
    M = cv2.getPerspectiveTransform(src, dst)
    V["Viewpoint (perspective)"] = (cv2.warpPerspective(img, M, (w, h)), M)
    return V

def sift_gt_precision(r, M, tol=5.0):
    if not r["good"]:
        return float("nan")
    src = np.float32([r["kpa"][m.queryIdx].pt for m in r["good"]]).reshape(-1, 1, 2)
    dst = np.float32([r["kpb"][m.trainIdx].pt for m in r["good"]]).reshape(-1, 1, 2)
    err = np.linalg.norm(cv2.perspectiveTransform(src, M) - dst, axis=2).ravel()
    return float((err < tol).mean())

def hog_repeatability(base_boxes, M, new_boxes):
    if not base_boxes:
        return float("nan")
    hit = 0
    for (x, y, w, h) in base_boxes:
        c = cv2.perspectiveTransform(np.float32([[[x + w / 2, y + h / 2]]]), M)[0, 0]
        if any(inside(c, nb) for nb in new_boxes):
            hit += 1
    return hit / len(base_boxes)

variants = make_variants(img1)
base_boxes, _, _ = hog_detect(img1, upscale=UPSCALE, score_thr=SCORE_THR)

rows = []
for name, (im, M) in variants.items():
    r = sift_match(img1, im)
    nb, ns, _ = hog_detect(im, upscale=UPSCALE, score_thr=SCORE_THR)
    rows.append({
        "Variant": name,
        "SIFT keypoints": len(r["kpb"]),
        "SIFT good matches": len(r["good"]),
        "SIFT RANSAC inliers": r["inliers"],
        "SIFT correct-match precision": round(sift_gt_precision(r, M), 2),
        "HOG detections": len(nb),
        "HOG repeatability": round(hog_repeatability(base_boxes, M, nb), 2),
    })
rob = pd.DataFrame(rows)
display(rob)
RESULTS["robustness"] = rows
rob.to_csv("figures/robustness_table.csv", index=False)

In [ ]:
x = np.arange(len(rob)); labels = rob["Variant"]
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].bar(x - 0.2, rob["SIFT good matches"], 0.4, label="Good matches", color="#4C78A8")
ax[0].bar(x + 0.2, rob["SIFT RANSAC inliers"], 0.4, label="RANSAC inliers", color="#54A24B")
ax[0].set_xticks(x); ax[0].set_xticklabels(labels, rotation=40, ha="right")
ax[0].set_title("Scenario 1 - SIFT: matches vs. Original"); ax[0].legend(); ax[0].set_ylabel("count")
ax[1].bar(x - 0.2, rob["HOG detections"], 0.4, label="Detections", color="#F58518")
ax[1].set_xticks(x); ax[1].set_xticklabels(labels, rotation=40, ha="right")
ax2 = ax[1].twinx(); ax2.plot(x, rob["HOG repeatability"], "o-", color="#E45756", label="Repeatability"); ax2.set_ylim(0, 1.05)
ax[1].set_title("Scenario 2 - HOG: detections and repeatability"); ax[1].set_ylabel("count"); ax2.set_ylabel("repeatability")
h1, l1 = ax[1].get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); ax[1].legend(h1 + h2, l1 + l2, loc="upper right")
plt.tight_layout(); save_fig("fig09_robustness_comparison"); plt.show()

# Visual examples of selected variants
pick = ["Darker (x0.5)", "Scale x0.5", "Rotate 20 deg", "Viewpoint (perspective)"]
fig, axes = plt.subplots(2, len(pick), figsize=(16, 7))
for j, name in enumerate(pick):
    im, M = variants[name]
    r = sift_match(img1, im)
    inl = [m for m, k in zip(r["good"], r["mask"]) if k] if r["mask"] is not None else []
    nb, ns, _ = hog_detect(im, upscale=UPSCALE, score_thr=SCORE_THR)
    vis = cv2.drawMatches(img1, r["kpa"], im, r["kpb"], sorted(inl, key=lambda m: m.distance)[:60], None,
                          matchColor=(0, 255, 0), flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    axes[0, j].imshow(to_rgb(vis)); axes[0, j].set_title(f"SIFT inliers: {name}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(to_rgb(draw_boxes(im, nb, ns))); axes[1, j].set_title(f"HOG: {name} ({len(nb)} det.)", fontsize=9); axes[1, j].axis("off")
plt.tight_layout(); save_fig("fig10_robustness_examples"); plt.show()

## 7. Side-by-side comparison
The first table summarises the measured numbers from this run; the second summarises the conceptual differences (for the discussion section of your report).

In [ ]:
s, h = RESULTS["sift"], RESULTS["hog"]
summary = pd.DataFrame([
    ["Output", "Keypoints + descriptors; matches between images", "Bounding boxes with SVM scores"],
    ["Image 1", f"{s['kp1']} keypoints", f"{h['n1']} pedestrians"],
    ["Image 2", f"{s['kp2']} keypoints", f"{h['n2']} pedestrians"],
    ["Cross-image result", f"{s['good']} ratio-test matches, {s['inliers']} RANSAC inliers ({s['inlier_ratio']:.0%})", "Detections found independently in each image"],
    ["Processing time (this run)", f"{s['t_detect_ms']:.0f} ms detect + {s['t_match_ms']:.0f} ms match", f"{h['t1_ms']:.0f} ms (Image 1), {h['t2_ms']:.0f} ms (Image 2)"],
], columns=["Metric", "SIFT (Scenario 1)", "HOG (Scenario 2)"])
display(summary)
summary.to_csv("figures/summary_table.csv", index=False)

concept = pd.DataFrame([
    ["Main purpose", "Find and match distinctive local points between images", "Detect a category of object (pedestrians) in an image"],
    ["Feature type", "Sparse local keypoint descriptors (128-D)", "Dense gradient-orientation histograms on a grid (3780-D per window)"],
    ["Needs training?", "No", "Yes (linear SVM trained on person / non-person windows)"],
    ["Scale invariance", "Built in (scale-space extrema)", "Not built in; needs an image pyramid"],
    ["Rotation invariance", "Built in (dominant orientation)", "No; assumes upright pedestrians"],
    ["Illumination", "Robust to moderate change (gradients, normalised descriptor)", "Robust (block normalisation), but weak contrast hurts"],
    ["Handles moving / articulated people", "Poor: matches need rigid, textured structure", "Good: designed for the upright human silhouette"],
    ["Typical failure", "Few keypoints on low-texture regions; pedestrians change pose between views", "Misses people smaller than 64x128 or non-upright; false positives on pole-like structures"],
    ["Cost", "Heavier descriptor computation and matching", "Cost grows with the number of windows (stride, pyramid scale)"],
], columns=["Aspect", "SIFT", "HOG"])
display(concept)
concept.to_csv("figures/conceptual_comparison.csv", index=False)

## 8. Export figures and results
Downloads a zip with all figures (for your Word report) and the CSV tables.  
**For the second submission file (PDF of the notebook):** after everything has run, use **File → Print** (Ctrl+P) → *Save as PDF*, or in a local Jupyter use *File → Save and Export Notebook As → PDF*.

In [ ]:
import json, shutil
with open("figures/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
shutil.make_archive("lab4_5_figures", "zip", "figures")
print(sorted(os.listdir("figures")))
try:
    from google.colab import files
    files.download("lab4_5_figures.zip")
except Exception as e:
    print("Not running in Colab - the zip is saved as lab4_5_figures.zip")